# Laboratorio: Optimización de Circuitos con ZX-Calculus

Exploración práctica de la optimización de circuitos mediante PyZX y Qiskit.

**Objetivos:**
- Demostrar la equivalencia de circuitos usando el ZX-Calculus
- Medir la reducción de CNOTs con optimización ZX sobre circuitos VQE/QAOA
- Comparar la fidelidad del circuito original vs optimizado
- Analizar la relación entre T-count y recursos de computación tolerante a fallos

**Nota:** Este laboratorio muestra las técnicas con y sin PyZX (que puede no estar instalado).

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from qiskit import QuantumCircuit
from qiskit.quantum_info import Operator, process_fidelity
from qiskit.circuit.library import EfficientSU2

# Verificar si PyZX está disponible
try:
    import pyzx as zx
    PYZX_AVAILABLE = True
    print(f'PyZX disponible: versión {zx.__version__}')
except ImportError:
    PYZX_AVAILABLE = False
    print('PyZX no instalado. Instalar con: pip install pyzx')
    print('Ejecutando análisis sin PyZX (demostraciones manuales)')

print('Qiskit importado correctamente')

## 1. Identidades del ZX-Calculus: verificación algebraica

In [ ]:
# Regla 1: Spider Fusion — Rz(α)·Rz(β) = Rz(α+β)
alpha, beta = np.pi/4, np.pi/3

qc1 = QuantumCircuit(1)
qc1.rz(alpha, 0)
qc1.rz(beta, 0)

qc2 = QuantumCircuit(1)
qc2.rz(alpha + beta, 0)

op1 = Operator(qc1)
op2 = Operator(qc2)
print(f'Spider Fusion: Rz({alpha:.3f})·Rz({beta:.3f}) = Rz({alpha+beta:.3f})?')
print(f'  Matrices iguales: {np.allclose(op1.data, op2.data)}')

# Regla 2: H·Z·H = X
qc_hzh = QuantumCircuit(1)
qc_hzh.h(0); qc_hzh.z(0); qc_hzh.h(0)

qc_x = QuantumCircuit(1)
qc_x.x(0)

print(f'\nH·Z·H = X?')
print(f'  Matrices iguales: {np.allclose(Operator(qc_hzh).data, Operator(qc_x).data)}')

# Regla 3: CNOT = (I⊗H)·CZ·(I⊗H)
qc_cnot = QuantumCircuit(2)
qc_cnot.cx(0, 1)

qc_cz_h = QuantumCircuit(2)
qc_cz_h.h(1); qc_cz_h.cz(0, 1); qc_cz_h.h(1)

print(f'\nCNOT = (I⊗H)·CZ·(I⊗H)?')
print(f'  Matrices iguales: {np.allclose(Operator(qc_cnot).data, Operator(qc_cz_h).data)}')

# Regla 4: ZZ se cancela
qc_zz = QuantumCircuit(1)
qc_zz.z(0); qc_zz.z(0)

qc_id = QuantumCircuit(1)  # identidad

print(f'\nZ·Z = I?')
print(f'  Matrices iguales: {np.allclose(Operator(qc_zz).data, Operator(qc_id).data)}')

## 2. Optimización Manual de un Circuito Redundante

In [ ]:
# Circuito con redundancias que el ZX-Calculus puede simplificar
qc_redundante = QuantumCircuit(2)
qc_redundante.h(0)
qc_redundante.cx(0, 1)
qc_redundante.cx(0, 1)  # CNOT·CNOT = I → se cancela
qc_redundante.h(0)
qc_redundante.h(0)      # H·H = I → se cancela
qc_redundante.cx(0, 1)

# Circuito optimizado equivalente
qc_optimizado = QuantumCircuit(2)
qc_optimizado.h(0)
qc_optimizado.cx(0, 1)

# Verificar equivalencia
op_red = Operator(qc_redundante)
op_opt = Operator(qc_optimizado)
son_iguales = np.allclose(op_red.data, op_opt.data)

print('Circuito redundante:')
print(qc_redundante.draw())
print(f'\nPuertas: {dict(qc_redundante.count_ops())}')
print(f'Profundidad: {qc_redundante.depth()}')

print('\nCircuito optimizado:')
print(qc_optimizado.draw())
print(f'Puertas: {dict(qc_optimizado.count_ops())}')
print(f'Profundidad: {qc_optimizado.depth()}')

print(f'\n¿Unitariamente equivalentes? {son_iguales}')
print(f'Reducción de CNOTs: {qc_redundante.count_ops()["cx"]} → {qc_optimizado.count_ops()["cx"]} ({100*(1 - qc_optimizado.count_ops()["cx"]/qc_redundante.count_ops()["cx"]):.0f}% menos)')

## 3. Optimización con PyZX (si está disponible)

In [ ]:
if PYZX_AVAILABLE:
    from qiskit.qasm2 import dumps
    
    # Circuito VQE parametrizado con valores fijos
    n_qubits = 4
    ansatz = EfficientSU2(n_qubits, reps=2, entanglement='linear')
    params = np.random.uniform(-np.pi, np.pi, ansatz.num_parameters)
    bound = ansatz.assign_parameters(params)
    
    # Estadísticas originales
    cx_original = bound.count_ops().get('cx', 0)
    depth_original = bound.depth()
    
    print(f'Circuito VQE EfficientSU2 ({n_qubits} qubits, reps=2):')
    print(f'  CNOTs originales: {cx_original}')
    print(f'  Profundidad original: {depth_original}')
    
    # Convertir a PyZX y optimizar
    qasm_str = dumps(bound)
    pyzx_circ = zx.Circuit.from_qasm(qasm_str)
    g = pyzx_circ.to_graph()
    
    print(f'\nGrafo ZX:')
    print(f'  Vértices: {g.num_vertices()}')
    print(f'  Aristas: {g.num_edges()}')
    
    zx.full_reduce(g)
    print(f'\nGrafo ZX simplificado:')
    print(f'  Vértices: {g.num_vertices()}')
    print(f'  Aristas: {g.num_edges()}')
    
    optimized = zx.extract_circuit(g)
    cx_optimized = optimized.count_2q_gates()
    
    print(f'\nResultados:')
    print(f'  CNOTs originales: {cx_original}')
    print(f'  CNOTs optimizados: {cx_optimized}')
    print(f'  Reducción: {100*(1 - cx_optimized/cx_original):.1f}%')

else:
    # Demostración con Qiskit transpiler
    from qiskit.transpiler.preset_passmanagers import generate_preset_pass_manager
    
    n_qubits = 4
    ansatz = EfficientSU2(n_qubits, reps=2, entanglement='linear')
    params = np.random.uniform(-np.pi, np.pi, ansatz.num_parameters)
    bound = ansatz.assign_parameters(params)
    
    cx_original = bound.count_ops().get('cx', 0)
    
    # Optimización con el transpilador de Qiskit (nivel 3)
    pm = generate_preset_pass_manager(optimization_level=3, basis_gates=['cx', 'u'])
    optimized_qc = pm.run(bound)
    cx_optimized = optimized_qc.count_ops().get('cx', 0)
    
    print(f'Optimización con Qiskit Transpiler (nivel 3):')
    print(f'  CNOTs originales: {cx_original}')
    print(f'  CNOTs tras optimización: {cx_optimized}')
    reduction = 100*(1 - cx_optimized/cx_original) if cx_original > 0 else 0
    print(f'  Reducción: {reduction:.1f}%')
    print(f'  (Para reducción ZX completa, instalar PyZX: pip install pyzx)')

## 4. T-count y Recursos de Corrección de Errores

In [ ]:
# El T-count (número de puertas T = Rz(π/4)) determina el coste en
# destilación de estados mágicos para la computación tolerante a fallos

def count_t_gates(circuit: QuantumCircuit) -> int:
    """Cuenta el número de puertas T (y T†) en un circuito."""
    ops = circuit.count_ops()
    return ops.get('t', 0) + ops.get('tdg', 0)

# Implementaciones de la puerta Toffoli con diferentes T-counts
# Estándar: 7 puertas T
tof_standard = QuantumCircuit(3)
tof_standard.h(2)
tof_standard.cx(1, 2)
tof_standard.tdg(2)
tof_standard.cx(0, 2)
tof_standard.t(2)
tof_standard.cx(1, 2)
tof_standard.tdg(2)
tof_standard.cx(0, 2)
tof_standard.t(1)
tof_standard.t(2)
tof_standard.h(2)
tof_standard.cx(0, 1)
tof_standard.t(0)
tof_standard.tdg(1)
tof_standard.cx(0, 1)

t_count_standard = count_t_gates(tof_standard)
print(f'Toffoli estándar: T-count = {t_count_standard}')

# Relación T-count con recursos de corrección de errores
print('\nCoste de T-count en corrección de errores:')
print(f'  Cada puerta T requiere ~distillation de estados mágicos')
print(f'  Con p_error = 1e-3 y T-count = {t_count_standard}:')
t_counts = [1, 7, 10, 100, 1000]
qubits_per_T = 15  # overhead de destilación de primer nivel
print(f'\n  {'T-count':<12} {'Qubits ancilla (estim.)'}' )
print('  ' + '-' * 35)
for T in t_counts:
    qubits_ancilla = T * qubits_per_T
    print(f'  {T:<12} {qubits_ancilla}')

print(f'\nEl ZX-Calculus puede reducir el T-count, reduciendo directamente')
print(f'el overhead de destilación de estados mágicos.')

## 5. Phase Gadgets: construcción y composición

Un **phase gadget** de ángulo α sobre n qubits implementa `exp(-i α/2 · Z⊗Z⊗...⊗Z)`.
Son el bloque fundamental de QAOA y de la exponenciación de operadores de Pauli.
En ZX-Calculus corresponden a un spider Z central con n alambres de entrada/salida.

In [ ]:
import scipy.linalg
from qiskit.quantum_info import SparsePauliOp

def phase_gadget(n_qubits: int, alpha: float) -> QuantumCircuit:
    """Phase gadget: exp(-i alpha/2 * Z^n). CNOT-tree + Rz + CNOT-tree inverso."""
    qc = QuantumCircuit(n_qubits)
    for i in range(n_qubits - 1):
        qc.cx(i, i + 1)
    qc.rz(alpha, n_qubits - 1)
    for i in range(n_qubits - 2, -1, -1):
        qc.cx(i, i + 1)
    return qc

pg4 = phase_gadget(4, np.pi/3)
print('Phase gadget (4 qubits, alpha=pi/3):')
print(pg4.draw('text'))
print(f'Puertas: {dict(pg4.count_ops())}')

# Verificar que implementa exp(-i alpha/2 ZZZZ)
alpha = np.pi/3
Z4 = SparsePauliOp('ZZZZ')
U_analitico = scipy.linalg.expm(-1j * alpha/2 * Z4.to_matrix())
U_circuito  = Operator(pg4).data
ratio = U_analitico.flat[0] / U_circuito.flat[0]
coinciden = np.allclose(U_analitico, ratio * U_circuito)
print(f'\nexp(-i alpha/2 ZZZZ) = phase_gadget(4, alpha)? {coinciden} (salvo fase global)')


In [ ]:
# Regla de fusión: PG(alpha) · PG(beta) = PG(alpha+beta)
# Esta es la spider fusion del ZX-Calculus aplicada a circuitos reales

alpha, beta = np.pi/4, np.pi/6
n = 3

qc_dos = QuantumCircuit(n)
qc_dos.compose(phase_gadget(n, alpha), inplace=True)
qc_dos.compose(phase_gadget(n, beta), inplace=True)

qc_fus = phase_gadget(n, alpha + beta)

equivalentes = np.allclose(Operator(qc_dos).data, Operator(qc_fus).data)
print(f'PG({alpha:.3f}) * PG({beta:.3f}) = PG({alpha+beta:.3f})?  {equivalentes}')
print(f'CNOTs antes de fusión: {qc_dos.count_ops().get("cx", 0)}')
print(f'CNOTs tras fusión:     {qc_fus.count_ops().get("cx", 0)}')
ahorro = qc_dos.count_ops().get('cx',0) - qc_fus.count_ops().get('cx',0)
print(f'Ahorro: {ahorro} CNOTs eliminados')

# Cancelación: PG(alpha) · PG(-alpha) = I
qc_cancel = QuantumCircuit(n)
qc_cancel.compose(phase_gadget(n, alpha), inplace=True)
qc_cancel.compose(phase_gadget(n, -alpha), inplace=True)
qc_id = QuantumCircuit(n)
print(f'\nPG(alpha) * PG(-alpha) = I?  {np.allclose(Operator(qc_cancel).data, Operator(qc_id).data)}')


## 6. Spider Fusion en circuitos QAOA multi-qubit

En QAOA, la capa de coste aplica un phase gadget ZZ por cada arista del grafo.
Si dos capas consecutivas tienen el mismo operador de Pauli, se fusionan:
QAOA_cost(gamma1) · QAOA_cost(gamma2) = QAOA_cost(gamma1+gamma2).

In [ ]:
def qaoa_cost_layer(n: int, gamma: float, edges) -> QuantumCircuit:
    """Capa de coste QAOA: producto de exp(-i gamma/2 Zi Zj) por arista."""
    qc = QuantumCircuit(n)
    for i, j in edges:
        qc.cx(i, j)
        qc.rz(gamma, j)
        qc.cx(i, j)
    return qc

n_nodes = 4
edges_sq = [(0,1),(1,2),(2,3),(3,0)]  # cuadrado
gamma = np.pi/4

qc_dos_capas = QuantumCircuit(n_nodes)
qc_dos_capas.compose(qaoa_cost_layer(n_nodes, gamma, edges_sq), inplace=True)
qc_dos_capas.compose(qaoa_cost_layer(n_nodes, gamma, edges_sq), inplace=True)

qc_fusionado = qaoa_cost_layer(n_nodes, 2*gamma, edges_sq)

equivalente = np.allclose(Operator(qc_dos_capas).data, Operator(qc_fusionado).data)
print('Spider fusion en QAOA:')
print(f'  Dos capas (gamma={gamma:.3f}): {qc_dos_capas.count_ops().get("cx",0)} CNOTs')
print(f'  Una capa fusionada (2*gamma={2*gamma:.3f}): {qc_fusionado.count_ops().get("cx",0)} CNOTs')
print(f'  Unitariamente equivalentes: {equivalente}')
ahorro_pct = 50  # siempre 50% cuando hay 2 capas iguales
print(f'  Ahorro: 50% de CNOTs por spider fusion')


## 7. Comparativa de profundidad antes/después de optimización ZX

Barremos circuitos QAOA con p = 1..6 capas y medimos el efecto de la optimización.

In [ ]:
from qiskit.transpiler.preset_passmanagers import generate_preset_pass_manager

resultados = []

for p_layers in range(1, 7):
    gammas = [np.pi/4 * (1 + 0.1*i) for i in range(p_layers)]
    betas  = [np.pi/8 * (1 + 0.05*i) for i in range(p_layers)]

    qc_full = QuantumCircuit(n_nodes)
    qc_full.h(range(n_nodes))
    for g, b in zip(gammas, betas):
        qc_full.compose(qaoa_cost_layer(n_nodes, g, edges_sq), inplace=True)
        for q in range(n_nodes):
            qc_full.rx(2*b, q)

    cx_antes = qc_full.count_ops().get('cx', 0)
    depth_antes = qc_full.depth()

    pm = generate_preset_pass_manager(optimization_level=3, basis_gates=['cx','rz','rx','h'])
    qc_opt = pm.run(qc_full)
    cx_despues = qc_opt.count_ops().get('cx', 0)
    depth_despues = qc_opt.depth()

    reduccion = 100*(cx_antes - cx_despues)/cx_antes if cx_antes > 0 else 0
    resultados.append({'p': p_layers, 'cx_a': cx_antes, 'cx_d': cx_despues,
                       'dep_a': depth_antes, 'dep_d': depth_despues, 'red': reduccion})

print(f"{'p':>3}  {'CNOT antes':>10}  {'CNOT opt':>9}  {'Reducción':>9}  {'Prof. antes':>11}  {'Prof. opt':>9}")
print('-' * 60)
for r in resultados:
    print(f"{r['p']:>3}  {r['cx_a']:>10}  {r['cx_d']:>9}  {r['red']:>8.1f}%  {r['dep_a']:>11}  {r['dep_d']:>9}")


In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))

p_vals = [r['p'] for r in resultados]
ax1.plot(p_vals, [r['cx_a'] for r in resultados], 'o-', color='#e74c3c', lw=2, label='Sin optimización')
ax1.plot(p_vals, [r['cx_d'] for r in resultados], 's-', color='#2ecc71', lw=2, label='Optimizado (nivel 3)')
ax1.set_xlabel('Capas p de QAOA'); ax1.set_ylabel('CNOTs')
ax1.set_title('CNOTs antes y después de optimización ZX')
ax1.legend(); ax1.grid(alpha=0.3)

ax2.bar(p_vals, [r['red'] for r in resultados], color='#3498db', alpha=0.8)
ax2.set_xlabel('Capas p de QAOA'); ax2.set_ylabel('Reducción de CNOTs (%)')
ax2.set_title('Reducción por optimización ZX')
ax2.set_ylim(0, 60); ax2.grid(axis='y', alpha=0.3)

plt.tight_layout()
plt.show()

print('Conclusion: phase gadget fusion + spider fusion reduce el CNOT-count de QAOA,')
print('disminuyendo el overhead de magic state distillation en hardware fault-tolerant.')
